## Legal LLM Test

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

PATH = "checkpoints/awq_models/law_llm_awq_w4a16"

# vLLM auto-detects compressed-tensors W4A16 from the checkpoint config
llm = LLM(model=PATH, dtype="auto", 
            gpu_memory_utilization=0.20, 
            max_model_len=4096)

tokenizer = AutoTokenizer.from_pretrained(PATH)

In [2]:
SYS = ("You are a legal assistant. Answer the question using ONLY the context. "
        "Cite the relevant clause. If the answer is not in the context, say so.")

sp = SamplingParams(temperature=0.0, max_tokens=256)   # greedy, deterministic

def answer(context, question):
    user = f"{SYS}\n\nContext:\n{context}\n\nQuestion: {question}"
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True
    )
    out = llm.generate([prompt], sp)
    return out[0].outputs[0].text.strip()

In [3]:
ctx = ("Section 7.2. Termination for Convenience. Either party may terminate this "
       "Agreement without cause upon ninety (90) days prior written notice to the "
       "other party. Section 7.3. Termination for Cause. A party may terminate "
       "immediately upon material breach that remains uncured for thirty (30) days.")

print(answer(ctx, "How much notice is required to terminate this agreement for convenience?"))

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.86it/s, est. speed input: 253.65 toks/s, output: 76.46 toks/s]

To terminate this agreement for convenience, a party must provide 90 days prior written notice to the other party. This information can be found in Section 7.2 of the provided context.


In [4]:
print(answer(ctx, "What is the governing law of this agreement?"))

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  8.03it/s, est. speed input: 1066.42 toks/s, output: 129.25 toks/s]

The governing law of this agreement is not explicitly mentioned in the provided context.


In [ ]:
ctx_irrelevant = ("Section 3.1. The Supplier shall deliver the Goods to the Buyer's "
                "warehouse at 14 Industrial Road by the Delivery Date.")
print(answer(ctx_irrelevant, "What are the elements of promissory estoppel?"))

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  7.02it/s, est. speed input: 677.63 toks/s, output: 127.04 toks/s]

The elements of promissorial estoppel are not mentioned in the provided context.


## Qwen Coder Test

In [1]:
# ── Cell 1: Imports ───────────────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import re, subprocess, sys, tempfile, os

/home/lisa/Arupreza/PRISM-RAG/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/lisa/Arupreza/PRISM-RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Cell 2: Config ────────────────────────────────────────────
BASE    = "/home/lisa/Arupreza/PRISM-RAG"
MODEL_PATH      = f"{BASE}/checkpoints/source_model/qwen_coder"
LORA_CHECKPOINT = f"{BASE}/checkpoints/awq_models/qwen_coder_grpo_lora/checkpoint-200"

SYSTEM = (
    "You are an expert Python programmer. Solve the problem below.\n"
    "Respond in EXACTLY this format:\n"
    "<reasoning>\nExplain your approach briefly.\n</reasoning>\n"
    "<code>\n```python\n# your complete solution here\n```\n</code>"
)

In [3]:
# ── Cell 3: Load Base + LoRA Adapter ─────────────────────────
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(LORA_CHECKPOINT)
tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print("Attaching LoRA adapter from checkpoint-200...")
model = PeftModel.from_pretrained(base_model, LORA_CHECKPOINT)
model.eval()

print(f"✓ Model loaded — device: {next(model.parameters()).device}")

Loading base model...


/home/lisa/Arupreza/PRISM-RAG/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Loading checkpoint shards: 100%|██████████| 14/14 [08:30<00:00, 36.44s/it]


Attaching LoRA adapter from checkpoint-200...
✓ Model loaded — device: cuda:0


In [4]:
# ── Cell 4: Generation Function ───────────────────────────────
def generate(problem: str, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": problem},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens (not the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [5]:
# ── Cell 5: Test Helper — Extract + Run Code ──────────────────
def extract_code(text: str) -> str:
    m = re.search(r"```python\s*\n(.*?)```", text, re.S)
    if m: return m.group(1).strip()
    return ""

def run_code(code: str, timeout: int = 8) -> dict:
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py",
                                    delete=False, dir="/tmp") as f:
        f.write(code); path = f.name
    try:
        r = subprocess.run(
            [sys.executable, path],
            capture_output=True, text=True, timeout=timeout,
        )
        return {"success": r.returncode == 0,
                "stdout": r.stdout[:300],
                "stderr": r.stderr[:300]}
    except subprocess.TimeoutExpired:
        return {"success": False, "stdout": "", "stderr": "TIMEOUT"}
    finally:
        os.remove(path)

def test(problem: str, test_code: str = "") -> None:
    print(f"Problem: {problem}\n{'─'*50}")
    response = generate(problem)
    print(response)

    code = extract_code(response)
    if not code:
        print("\n⚠ No code block found in response.")
        return

    print(f"\n{'─'*50}\nExtracted code:\n{code}")

    if test_code:
        full = code + "\n\n" + test_code
        result = run_code(full)
        status = "✅ PASS" if result["success"] else "❌ FAIL"
        print(f"\nTest result: {status}")
        if not result["success"]:
            print(f"Error: {result['stderr'][:200]}")
    else:
        # Just run the code alone to check for syntax/runtime errors
        result = run_code(code)
        print(f"\nRun result: {'✅ No error' if result['success'] else '❌ Error'}")
        if result["stdout"]: print(f"Output: {result['stdout']}")
        if not result["success"]: print(f"Error: {result['stderr'][:200]}")

In [6]:
# ── Cell 6: Test 1 — Easy (String Reversal) ───────────────────
test(
    problem="Write a Python function to reverse a string.",
    test_code="""
assert reverse_string("hello") == "olleh"
assert reverse_string("") == ""
assert reverse_string("a") == "a"
print("All tests passed!")
"""
)

Problem: Write a Python function to reverse a string.
──────────────────────────────────────────────────
<reasoning>
To reverse a string in Python, we can use slicing. Slicing allows us to create a new string that is a reversed version of the original string by specifying a step of -1, which means we start from the end towards the beginning.
</reasoning>
<code>
```python
def reverse_string(s):
    return s[::-1]
```
</code>

──────────────────────────────────────────────────
Extracted code:
def reverse_string(s):
    return s[::-1]

Test result: ✅ PASS


In [7]:
# ── Cell 7: Test 2 — Medium (Two Sum) ────────────────────────
test(
    problem="Write a function two_sum(nums, target) that returns indices of the two numbers that add up to target.",
    test_code="""
assert two_sum([2, 7, 11, 15], 9) == [0, 1]
assert two_sum([3, 2, 4], 6) == [1, 2]
assert two_sum([3, 3], 6) == [0, 1]
print("All tests passed!")
"""
)

Problem: Write a function two_sum(nums, target) that returns indices of the two numbers that add up to target.
──────────────────────────────────────────────────
<reasoning>
To solve the two_sum problem, we can use a hash map (dictionary in Python) to store the numbers we have seen so far along with their indices. As we iterate through the list, for each number, we calculate its complement with respect to the target (i.e., `target - num`). If this complement is already in the dictionary, it means we have found the two numbers that add up to the target, and we can return their indices. If not, we add the current number and its index to the dictionary and continue. This approach ensures that we only pass through the list once, achieving a time complexity of O(n).
</reasoning>
<code>
```python
def two_sum(nums, target):
    num_to_index = {}
    for index, num in enumerate(nums):
        complement = target - num
        if complement in num_to_index:
            return [num_to_index[comp

In [8]:
# ── Cell 8: Test 3 — Hard (LRU Cache) ────────────────────────
test(
    problem="""Implement an LRU (Least Recently Used) cache with the following operations:
- LRUCache(capacity): Initialize with positive capacity
- get(key): Return the value if key exists, else -1
- put(key, value): Update or insert the value. Evict LRU key if capacity exceeded.""",
    test_code="""
lru = LRUCache(2)
lru.put(1, 1)
lru.put(2, 2)
assert lru.get(1) == 1
lru.put(3, 3)           # evicts key 2
assert lru.get(2) == -1
lru.put(4, 4)           # evicts key 1
assert lru.get(1) == -1
assert lru.get(3) == 3
assert lru.get(4) == 4
print("All tests passed!")
"""
)

Problem: Implement an LRU (Least Recently Used) cache with the following operations:
- LRUCache(capacity): Initialize with positive capacity
- get(key): Return the value if key exists, else -1
- put(key, value): Update or insert the value. Evict LRU key if capacity exceeded.
──────────────────────────────────────────────────
<reasoning>
To implement an LRU (Least Recently Used) Cache, we need a data structure that supports efficient insertion, deletion, and lookup operations. The `collections.OrderedDict` in Python is ideal for this purpose because it maintains the order of keys based on their usage (insertion order). Here's how we can use it:

1. **Initialization**: We initialize the cache with a given capacity and an empty `OrderedDict`.
2. **Get Operation**: When we retrieve a value using a key, we check if the key exists in the cache. If it does, we move the key to the end of the `OrderedDict` (indicating it was recently used) and return its value. If it doesn't exist, we return -1

In [9]:
# ── Cell 9: Compare Base vs Fine-Tuned ───────────────────────
# Load base WITHOUT LoRA to compare
print("Loading base model (no LoRA) for comparison...")
base_only = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, quantization_config=bnb,
    device_map="auto", torch_dtype=torch.bfloat16,
)
base_only.eval()

PROBLEM = "Write a Python function to find the longest palindromic substring."

def generate_with(m, problem):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": problem},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=512, temperature=0.7,
                            do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("\n" + "="*60)
print("BASE MODEL (no fine-tuning):")
print("="*60)
base_response = generate_with(base_only, PROBLEM)
print(base_response)

print("\n" + "="*60)
print("FINE-TUNED MODEL (checkpoint-200):")
print("="*60)
ft_response = generate_with(model, PROBLEM)
print(ft_response)

Loading base model (no LoRA) for comparison...


Loading checkpoint shards: 100%|██████████| 14/14 [02:13<00:00,  9.55s/it]



BASE MODEL (no fine-tuning):
<reasoning>
To find the longest palindromic substring, we can use a center expansion technique. The idea is to consider each character (and each pair of consecutive characters) as the center of a potential palindrome and expand outwards as long as the substring remains a palindrome. This approach works because a palindrome mirrors around its center. We keep track of the longest palindrome found during these expansions.
</reasoning>
<code>
```python
def longest_palindromic_substring(s: str) -> str:
    if not s or len(s) == 1:
        return s

    start, end = 0, 0

    for i in range(len(s)):
        len1 = expand_around_center(s, i, i)   # Odd length palindromes
        len2 = expand_around_center(s, i, i+1) # Even length palindromes
        max_len = max(len1, len2)

        if max_len > end - start:
            start = i - (max_len - 1) // 2
            end = i + max_len // 2

    return s[start:end+1]

def expand_around_center(s: str, left: int, right

In [10]:
# ── Cell 10: Batch Evaluation — Pass Rate ─────────────────────
problems = [
    {
        "q": "Write a function to check if a number is prime.",
        "test": "assert is_prime(2)==True\nassert is_prime(4)==False\nassert is_prime(17)==True\nprint('OK')"
    },
    {
        "q": "Write a function to flatten a nested list.",
        "test": "assert flatten([1,[2,[3]],4])==[1,2,3,4]\nprint('OK')"
    },
    {
        "q": "Write a function to count word frequency in a string.",
        "test": "assert word_freq('a b a')['a']==2\nprint('OK')"
    },
    {
        "q": "Write a binary search function.",
        "test": "assert binary_search([1,3,5,7,9],5)==2\nassert binary_search([1,3,5],6)==-1\nprint('OK')"
    },
    {
        "q": "Write a function to check if a string is a valid palindrome (ignoring spaces and case).",
        "test": "assert is_palindrome('A man a plan a canal Panama')==True\nassert is_palindrome('hello')==False\nprint('OK')"
    },
]

passed = 0
for i, p in enumerate(problems):
    response = generate(p["q"])
    code = extract_code(response)
    result = run_code(code + "\n\n" + p["test"]) if code else {"success": False}
    status = "✅" if result["success"] else "❌"
    print(f"{status} Problem {i+1}: {p['q'][:50]}...")
    if result["success"]: passed += 1

print(f"\nPass rate: {passed}/{len(problems)} = {passed/len(problems)*100:.0f}%")

✅ Problem 1: Write a function to check if a number is prime....
✅ Problem 2: Write a function to flatten a nested list....
❌ Problem 3: Write a function to count word frequency in a stri...
✅ Problem 4: Write a binary search function....
✅ Problem 5: Write a function to check if a string is a valid p...

Pass rate: 4/5 = 80%
